# VFX Production Intelligence Dashboard — Projects Cleaning

Work through this table from top to bottom. The context and rules below are inherited from earlier milestones.

## What we already know

- **Business name:** Projects
- **Purpose:** Stores project-level production information used to analyze client work, schedules, delivery performance, planned scope, budgeted labor, production leadership, and technical delivery requirements.
- **One row represents:** One VFX production project.
- **Expected primary key:** `project_id`

### Relationships
- client_id → clients.client_id

### Field rules from the approved data dictionary

| Field | Definition | Expected type | Null rule | Key role | Uniqueness | Allowed values / format | Cleaning expectation |
|---|---|---|---|---|---|---|---|
| `project_id` | Unique identifier assigned to each VFX production project. | Identifier text | No nulls allowed | Primary key | Required | PRJ-####: uppercase PRJ, a hyphen, and four digits. | Remove the exact duplicate row, retain one canonical project record, standardize the identifier, and verify uniqueness and completeness. |
| `client_id` | Identifier of the client account that commissioned or owns the production project. | Identifier text | No nulls allowed | Foreign key | Not required | CL-###: uppercase CL, a hyphen, and three digits. | Trim and standardize identifiers; correct only confirmed client assignments and quarantine unresolved orphan references from relationship-dependent analysis. |
| `project_name` | Business-facing title used to identify the production project. | Text | No nulls allowed | Not a key | Not required | Trimmed project title in consistent display casing. | Trim whitespace and remove repeated names only as part of exact-row deduplication; project_id remains the identifier. |
| `project_type` | Category describing the project’s production format or type of work. | Category text | No nulls allowed | Not a key | Not required | Commercial; Episodic; Feature Film; Game Cinematic; Independent Feature; Streaming Series; Trailer. | Trim whitespace and standardize to the approved project-type labels. |
| `start_date` | Date on which production work for the project officially began or was scheduled to begin. | Date | No nulls allowed | Not a key | Not required | YYYY-MM-DD. | Parse valid source formats, store as a true date, and display consistently as YYYY-MM-DD. |
| `target_delivery_date` | Planned date by which the project is expected to be delivered to the client. | Date | No nulls allowed | Not a key | Not required | YYYY-MM-DD and on or after start_date. | Preserve valid dates, investigate and correct the impossible sequence, and validate target_delivery_date >= start_date. |
| `actual_delivery_date` | Date on which the completed project was actually delivered to the client. | Date | Conditionally nullable | Not a key | Not required | YYYY-MM-DD, or null while the project has not been delivered. | Parse valid dates, preserve legitimate nulls for unfinished projects, and require a date when the standardized status is Completed. |
| `status` | Current lifecycle stage of the production project. | Category text | No nulls allowed | Not a key | Not required | Active; In Progress; In Delivery; Completed. | Trim whitespace and map variants to the four approved lifecycle statuses. |
| `priority` | Production priority assigned to the project for scheduling and resource-planning decisions. | Category text | No nulls allowed | Not a key | Not required | Medium; High; Critical. | Trim whitespace and standardize capitalization to the approved values. |
| `planned_shot_count` | Number of VFX shots planned within the project’s approved production scope. | Integer | No nulls allowed | Not a key | Not required | Positive whole number. | Preserve valid positive integers and investigate missing, zero, negative, or fractional values. |
| `budget_hours` | Total labor hours budgeted for completion of the project. | Decimal | No nulls allowed | Not a key | Not required | Positive numeric value. | Remove formatting text, convert to decimal, obtain confirmed replacements for missing or invalid budgets, and reject non-positive values before analysis. |
| `producer` | Producer responsible for coordinating the project schedule, staffing, client communication, and delivery. | Text | No nulls allowed | Not a key | Not required | Person name in First Last format. | Trim whitespace and standardize name presentation while preserving legitimate repetition. |
| `vfx_supervisor` | VFX supervisor responsible for the project’s creative and technical visual-effects oversight. | Text | No nulls allowed | Not a key | Not required | Recorded person name, commonly initial and surname. | Trim whitespace and preserve the established name format. |
| `fps` | Frame rate at which the project is produced and delivered. | Decimal | No nulls allowed | Not a key | Not required | 23.976; 24; 25; 29.97. | Preserve valid positive frame rates as decimals and investigate unsupported values. |
| `resolution` | Target image resolution and framing specification for project delivery. | Category text | No nulls allowed | Not a key | Not required | 2K Flat; 2K Scope; 4K DCI; 4K UHD. | Trim whitespace and standardize to the approved resolution labels. |
| `delivery_format` | File format and bit depth required for the project’s final image delivery. | Category text | No nulls allowed | Not a key | Not required | DPX 10-bit; EXR 16-bit; EXR 32-bit; ProRes 4444. | Trim whitespace and standardize to the approved delivery-format labels. |

### Known issues and approved decisions
- project_id: The raw table contains 16 rows and 15 distinct project IDs; PRJ-1005 appears twice as an exact duplicate.
- project_id: project_id remains the intended primary key. The repeated value is an exact duplicate raw record.
- client_id: PRJ-1003 contains CL-004 with surrounding whitespace, and PRJ-1011 references nonexistent client CL-999.
- client_id: client_id remains a required foreign key. The whitespace variant is recoverable; CL-999 must be corrected or quarantined.
- project_name: Starforge Cinematic repeats only because the complete PRJ-1005 row is duplicated.
- start_date: The raw text field contains mixed date formats, including 15-Jan-2026.
- start_date: The field is logically a date; mixed source formats explain the observed text type and will be standardized.
- target_delivery_date: One project has a target date earlier than its start date.
- target_delivery_date: The date-order violation is a raw-data defect and must be corrected before schedule analysis.
- actual_delivery_date: The raw text field contains mixed formats and five blanks associated with unfinished projects.
- actual_delivery_date: Nulls are valid for unfinished projects; mixed date formats explain the observed text type and will be standardized.
- status: The raw field includes variants such as complete and In-Progress.
- status: Multiple raw labels represent the same project states and will be standardized.
- budget_hours: The raw text field contains one missing value, one negative value (-250), and one formatted value (2,551.0 hrs).
- budget_hours: The field is expected to be numeric and required; raw formatting, a blank, and a negative value explain the text type and require remediation.
- producer: Repeated names are valid because one producer may oversee multiple projects.
- vfx_supervisor: Repeated names are valid because one supervisor may oversee multiple projects.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r'C:/Users/Dan/Documents/MEGA/Dev/GitHub/career-accelerator/projects/project-01-vfx-production-intelligence')
TABLE_NAME = 'projects'
RAW_PATH = PROJECT_DIR / r'data/raw/csv/raw_projects.csv'
PROCESSED_PATH = PROJECT_DIR / r'data/processed/csv/projects.csv'
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

if RAW_PATH.suffix.lower() == '.csv':
    raw_df = pd.read_csv(RAW_PATH)
elif RAW_PATH.suffix.lower() == '.parquet':
    raw_df = pd.read_parquet(RAW_PATH)
else:
    raise ValueError(f'Add the appropriate pandas reader for {RAW_PATH.suffix}')

clean_df = raw_df.copy()
print(f'{TABLE_NAME}: {len(raw_df):,} raw rows, {len(raw_df.columns)} columns')
raw_df.head()


## 1. Profile the raw table

- Confirm the source row count and column names.
- Measure missing values by field.
- Check exact duplicate rows.
- Test uniqueness and nulls for `project_id`.
- Review observed categories and parsing problems.
- Compare findings with the dictionary rules above before changing data.

In [ ]:
# Write the profiling checks for this table here.
# Keep the outputs that justify your cleaning decisions.


## 2. Apply the approved cleaning plan

Transform `clean_df` without modifying `raw_df`. Follow the field-level expectations above. Document any treatment that differs from the approved dictionary.

In [ ]:
# Write this table's cleaning transformations here.
# Example structure only: clean_df = clean_df.copy()


## 3. Validate the processed result

- Required columns are still present.
- Expected logical types can be produced consistently.
- Required fields do not contain unresolved nulls.
- Allowed values and formats match the dictionary.
- Invalid negative, out-of-range, or impossible values are resolved or documented.
- `project_id` is non-null and unique.
- Foreign-key and relationship exceptions are measured and documented.

In [ ]:
# Write the before-and-after validation checks here.
# The checks should fail visibly when an unresolved issue remains.


## 4. Export the reviewed table

After validation, save the reviewed result to `data/processed/csv/projects.csv`. The Data Cleaning Studio will discover and validate the file.

In [ ]:
# Run only after the table has passed your validation checks.
clean_df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved {len(clean_df):,} rows to {PROCESSED_PATH}')


## Cleaning summary

<!-- Describe what changed, why each important decision was appropriate, how many records were affected, and any remaining exception that a later milestone must know about. -->
